In [1]:
import pandas as pd
from datetime import datetime

In [2]:
def load_stock(path):
    # Load the CSV file
    df = pd.read_csv(path)  # replace with your actual file name

    # Handle missing values: drop rows with any missing data
    df.dropna(inplace=True)

    # Sort data by supplier
    df = df.sort_values(by='supplier')
    
    return df

In [3]:
class Drug:
    def __init__(self, filepath):
        # Store the dataframe as an instance attribute
        self.drugs = pd.read_csv(filepath)
        # Convert expiry_date to datetime for proper comparisons
        self.drugs['expiry_date'] = pd.to_datetime(self.drugs['expiry_date'])
        # You can also call a summary method here if needed
        # self.summary = self.drug_summary()   # if you define drug_summary
        self.total_drugs = len(self.drugs)
        
    def price_summary(self):
        """Return most expensive and cheapest drug details."""
        df = self.drugs
        most_expensive = df.loc[df['unit_price_usd'].idxmax()]
        cheapest = df.loc[df['unit_price_usd'].idxmin()]
        return {
            "most_expensive": most_expensive.to_dict(),  # convert Series to dict
            "cheapest": cheapest.to_dict()
        }

    def supplier_summary(self):
        df = self.drugs
        supplier_dict = {}
        for supplier, group in df.groupby('supplier'):
            supplier_dict[supplier] = {
                'list_of_drugs': group['drug_name'].tolist(),
                'total_number': int(group['quantity_in_stock'].sum())
            }
        counts = {
            'form': df['form'].value_counts().to_dict(),
            'category': df['category'].value_counts().to_dict(),
            'supplier': df['supplier'].value_counts().to_dict()
        }
        return {'supplier_details': supplier_dict, 'counts': counts}
    
    def total_inventory_value(self, by_supplier=False):
        """
        Calculate the total inventory value (quantity * unit_price).
        
        Args:
            by_supplier (bool): If True, return a dictionary with values per supplier.
        
        Returns:
            float or dict: Total value (float) if by_supplier=False,
                        otherwise dict {supplier: total_value}.
        """
        df = self.drugs
        df['total_value'] = df['quantity_in_stock'] * df['unit_price_usd']
        
        if by_supplier:
            return df.groupby('supplier')['total_value'].sum().to_dict()
        else:
            return df['total_value'].sum()
        
    def description(self):
        """Return a string describing the Drug class and its data."""
        return (
            f"Drug Class: Manages pharmaceutical drug data.\n"
            f"Loaded {self.total_drugs} drug records.\n"
            f"Available methods: price_summary(), supplier_summary().\n"
            f"Use this class to analyze drug prices and supplier information."
        )


In [4]:
class Inventory:
    
    def __init__(self, df):
        # Store the dataframe as an instance attribute so all methods can access it
        self.df = df.copy()  # copy to avoid modifying original
        # Ensure expiry_date is datetime for consistent comparisons
        self.df['expiry_date'] = pd.to_datetime(self.df['expiry_date'])
        # Store total items as an attribute (optional)
        self.total_items = self.df['quantity_in_stock'].sum()
        self.total_drugs = len(self.df)
    
    
    
    def get_low_stock(self):
        
        df = self.df
        return df[(df['quantity_in_stock'] <= df['reorder_level']) & 
                  (df['reorder_level'] - df['quantity_in_stock'] <= 15)]
    
    def get_re_stock(self):
        
        df = self.df
        return df[df['quantity_in_stock'] <= df['reorder_level'] + 20]
    
    def stock_summary(self):
        """
        Return a dictionary with details of most/least stock and newest/oldest expiry.
        """
        df = self.df
        most_stock = df.loc[df['quantity_in_stock'].idxmax()].to_dict()
        least_stock = df.loc[df['quantity_in_stock'].idxmin()].to_dict()
        latest_expiry = df.loc[df['expiry_date'].idxmax()].to_dict()
        oldest_expiry = df.loc[df['expiry_date'].idxmin()].to_dict()

        return {
            "most_stock": most_stock,
            "least_stock": least_stock,
            "latest_expiry": latest_expiry,
            "oldest_expiry": oldest_expiry
        }
    
    def check_expiry(self):
        
        today = pd.Timestamp.now().normalize()
        expired = self.df[self.df["expiry_date"] < today]
        return expired

    def near_expiry(self, days=30):
        
        today = pd.Timestamp.now().normalize()
        threshold = today + pd.Timedelta(days=days)
        near_exp = self.df[(self.df['expiry_date'] > today) & 
                        (self.df['expiry_date'] <= threshold)]
        return near_exp

    def expiry_loss_forecast(self, days=90):
        
        expiring = self.near_expiry(days)
        
        if expiring.empty:
            return 0.0
        # Compute total value (quantity * unit price)
        total_value = (expiring['quantity_in_stock'] * expiring['unit_price_usd']).sum()
        return total_value
    
    def description(self):
        return (
            f"Inventory Class: Manages drug inventory operations.\n"
            f"Currently tracking {self.total_drugs} drug types with {self.total_items} total items in stock.\n"
            f"Available methods: check_expiry(), get_low_stock(), get_re_stock(), stock_summary(), near_expiry(), description()."
        )
    
    def restock_report(self, filename="restock_report.txt"):
        
        lines = []
        lines.append(f"Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
        lines.append("")

        # Basic counts (from instance attributes)
        lines.append(f"Total drug types: {self.total_drugs}")
        lines.append(f"Total items in stock: {self.total_items}")
        lines.append("")

        # Total inventory value (compute directly)
        total_value = (self.df['quantity_in_stock'] * self.df['unit_price_usd']).sum()
        lines.append(f"Total inventory value: ${total_value:,.2f}")
        lines.append("")

        # Expired drugs (using check_expiry)
        expired = self.check_expiry()
        lines.append(f"Expired drugs: {len(expired)}")
        if not expired.empty:
            lines.append("  " + ", ".join(expired['drug_name'].tolist()))
        lines.append("")

        # Low stock (within 15 below reorder) using get_low_stock
        low_stock = self.get_low_stock()
        lines.append(f"Low stock drugs (within 15 below reorder): {len(low_stock)}")
        if not low_stock.empty:
            for _, row in low_stock.iterrows():
                lines.append(f"  {row['drug_name']}: {row['quantity_in_stock']} (reorder at {row['reorder_level']})")
        lines.append("")

        # Restock soon (≤ reorder + 20) using get_re_stock
        restock = self.get_re_stock()
        lines.append(f"Drugs needing restock soon (≤ reorder+20): {len(restock)}")
        if not restock.empty:
            for _, row in restock.iterrows():
                lines.append(f"  {row['drug_name']}: {row['quantity_in_stock']} (reorder at {row['reorder_level']})")
        lines.append("")

        # Expiry loss forecasts using expiry_loss_forecast
        lines.append("Expiry loss forecast (value of drugs expiring within...):")
        for days in [30, 60, 90]:
            loss = self.expiry_loss_forecast(days)
            lines.append(f"  Next {days} days: ${loss:,.2f}")
        lines.append("")

        # Top 5 most stocked drugs
        top_stock = self.df.nlargest(5, 'quantity_in_stock')[['drug_name', 'quantity_in_stock']]
        lines.append("Top 5 most stocked drugs:")
        for _, row in top_stock.iterrows():
            lines.append(f"  {row['drug_name']}: {row['quantity_in_stock']} units")
        lines.append("")

        # Top 5 most valuable drugs (by current stock value)
        # Add temporary column without altering original df
        df_temp = self.df.copy()
        df_temp['stock_value'] = df_temp['quantity_in_stock'] * df_temp['unit_price_usd']
        top_value = df_temp.nlargest(5, 'stock_value')[['drug_name', 'stock_value']]
        lines.append("Top 5 most valuable drugs (by current stock value):")
        for _, row in top_value.iterrows():
            lines.append(f"  {row['drug_name']}: ${row['stock_value']:,.2f}")
        lines.append("")

        lines.append("=" * 60)

        # Write to file with UTF‑8 encoding
        with open(filename, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))

        # Print to console
        # print(f"\nRestock report saved to {filename}")
        return "\n".join(lines)
        

    def expiry_report(self, days=90, filename="expiry_report.txt"):
        
        lines = []
        lines.append(f"Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
        lines.append(f"Forecast window: {days} days")
        lines.append("")

        # ---------- Expired drugs ----------
        expired = self.check_expiry()
        lines.append("EXPIRED DRUGS")
        lines.append("-" * 40)
        if expired.empty:
            lines.append("  No expired drugs.")
        else:
            lines.append(f"  Total expired items: {len(expired)}")
            lines.append("  List:")
            for _, row in expired.iterrows():
                lines.append(f"    {row['drug_name']} (expired on {row['expiry_date'].date()}) – {row['quantity_in_stock']} units")
        lines.append("")

        # ---------- Near‑expiry drugs (within `days`) ----------
        near = self.near_expiry(days)
        lines.append(f"DRUGS EXPIRING WITHIN {days} DAYS")
        lines.append("-" * 40)
        if near.empty:
            lines.append(f"  No drugs expire within the next {days} days.")
        else:
            lines.append(f"  Total items: {len(near)}")
            lines.append("  List by expiry date:")
            # Sort by expiry date for clarity
            near_sorted = near.sort_values('expiry_date')
            for _, row in near_sorted.iterrows():
                lines.append(f"    {row['drug_name']} – expires {row['expiry_date'].date()} – {row['quantity_in_stock']} units")
        lines.append("")

        # ---------- Loss forecast ----------
        loss = self.expiry_loss_forecast(days)
        lines.append("LOSS FORECAST")
        lines.append("-" * 40)
        lines.append(f"  Total value of drugs expiring within {days} days: ${loss:,.2f}")
        lines.append("")

        # ---------- Breakdown by category (optional) ----------
        if not near.empty:
            # Add temporary value column
            near_with_value = near.copy()
            near_with_value['value'] = near_with_value['quantity_in_stock'] * near_with_value['unit_price_usd']

            # By category
            cat_sum = near_with_value.groupby('category')['value'].sum().sort_values(ascending=False)
            lines.append("EXPIRING VALUE BY CATEGORY")
            lines.append("-" * 40)
            for category, val in cat_sum.items():
                lines.append(f"  {category}: ${val:,.2f}")
            lines.append("")

            # Top 5 most valuable expiring drugs
            top_value = near_with_value.nlargest(5, 'value')[['drug_name', 'value']]
            lines.append("TOP 5 MOST VALUABLE EXPIRING DRUGS")
            lines.append("-" * 40)
            for _, row in top_value.iterrows():
                lines.append(f"  {row['drug_name']}: ${row['value']:,.2f}")
            lines.append("")

        # Write to file with UTF‑8 encoding
        with open(filename, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))

        #DRUG INVENTORY RESTOCK REPORT
       # print(f"\nExpiry report saved to {filename}")
        return "\n".join(lines)
        

In [ ]:
class DrugItem:
    

    def __init__(self, drug_series):
        
        self.drug_id = drug_series['drug_id']
        self.drug_name = drug_series['drug_name']
        self.category = drug_series['category']
        self.quantity = int(drug_series['quantity_in_stock'])
        self.reorder_level = int(drug_series['reorder_level'])
        self.unit_price = float(drug_series['unit_price_usd'])
        self.form = drug_series['form']
        self.supplier = drug_series['supplier']

        # Parse expiry date
        expiry = drug_series['expiry_date']
        if isinstance(expiry, str):
            self.expiry_date = pd.to_datetime(expiry).date()
        elif isinstance(expiry, pd.Timestamp):
            self.expiry_date = expiry.date()
        else:
            self.expiry_date = expiry  # assume it's already date or None

   
    def is_expired(self):
        """Return True if the drug has already expired."""
        if self.expiry_date is None:
            return False
        return self.expiry_date < datetime.now().date()

    def days_to_expiry(self):
        """Return number of days until expiry (negative if expired)."""
        if self.expiry_date is None:
            return None
        return (self.expiry_date - datetime.now().date()).days

    def needs_reorder(self):
        
        if self.is_expired():
            return False
        return self.quantity <= self.reorder_level

    def value(self):
        """Return total monetary value of this drug in stock."""
        return self.quantity * self.unit_price

    def expiry_status(self):
        """Return a sentence describing the expiry status."""
        if self.expiry_date is None:
            return "Expiry date unknown."
        days = self.days_to_expiry()
        if days < 0:
            return f"Expired {abs(days)} days ago."
        elif days == 0:
            return "Expires today."
        else:
            return f"Expires in {days} days."

    def reorder_status(self):
        """Return a sentence describing whether reorder is needed."""
        if self.is_expired():
            return "Reorder needed: Drug is expired."
        if self.needs_reorder():
            return f"Reorder needed: current stock ({self.quantity}) is at or below reorder level ({self.reorder_level})."
        else:
            return f"Stock is adequate ({self.quantity} units, reorder at {self.reorder_level})."

    def value_description(self):
        """Return a sentence describing the total stock value."""
        return f"Total value of {self.drug_name} in stock: ${self.value():,.2f}."

    def stock_status(self):
        
        if self.is_expired():
            return "Expired"
        if self.needs_reorder():
            return "Low Stock (below reorder level)"
        if self.quantity <= self.reorder_level + 20:
            return "Adequate (close to reorder)"
        return "Well Stocked"

    def full_report(self):
        """Return a multi‑line string with all key information."""
        lines = [
            f"Drug: {self.drug_name} ({self.drug_id})",
            f"Category: {self.category} | Form: {self.form} | Supplier: {self.supplier}",
            f"Stock: {self.quantity} units (reorder at {self.reorder_level})",
            self.expiry_status(),
            self.reorder_status(),
            self.value_description(),
            f"Status: {self.stock_status()}"
        ]
        return "\n".join(lines)

    def info_dict(self):
        """Return a dictionary of all drug attributes (raw values)."""
        return {
            'drug_id': self.drug_id,
            'drug_name': self.drug_name,
            'category': self.category,
            'quantity': self.quantity,
            'reorder_level': self.reorder_level,
            'unit_price': self.unit_price,
            'expiry_date': self.expiry_date,
            'form': self.form,
            'supplier': self.supplier,
            'expired': self.is_expired(),
            'days_to_expiry': self.days_to_expiry(),
            'needs_reorder': self.needs_reorder(),
            'stock_value': self.value()
        }

In [11]:

def generate_restock_report(Inventory):

    return Inventory.restock_report()


In [12]:
def generate_expiry_report(Inventory):

    return Inventory.expiry_report()


In [15]:
# ==================== CONFIGURATION ====================
# Change this to the path of your CSV file
filepath = "06_drug_inventory.csv"   # <-- UPDATE WITH YOUR FILE PATH
# =======================================================

# ==================== Import or copy your code here ====================
# If your code is in a separate module, uncomment the import below.
# from your_module import load_stock, Drug, Inventory, DrugItem, generate_restock_report, generate_expiry_report

# Otherwise, paste the full code from your previous message here.
# (For brevity, I'm assuming the code is already in the environment.)

# ==================== Helper to print section headers ====================
def print_header(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

# ==================== Test load_stock ====================
def test_load_stock(filepath):
    print_header("Testing load_stock")
    try:
        df = load_stock(filepath)
        print(f"File loaded successfully. Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print("Missing values after load_stock:", df.isnull().sum().sum())
        print("Data sorted by supplier? (check first few):")
        print(df[['supplier', 'drug_name']].head())
    except Exception as e:
        print(f"Error in load_stock: {e}")

# ==================== Test Drug class ====================
def test_drug_class(filepath):
    print_header("Testing Drug Class")
    try:
        drug_mgr = Drug(filepath)
        print("Drug instance created. Total drugs:", drug_mgr.total_drugs)

        print("\n--- price_summary() ---")
        price_sum = drug_mgr.price_summary()
        print("Most expensive:", price_sum['most_expensive']['drug_name'], 
              "(${:.2f})".format(price_sum['most_expensive']['unit_price_usd']))
        print("Cheapest:", price_sum['cheapest']['drug_name'],
              "(${:.2f})".format(price_sum['cheapest']['unit_price_usd']))

        print("\n--- supplier_summary() ---")
        supp_sum = drug_mgr.supplier_summary()
        print("Supplier details:")
        for supplier, details in supp_sum['supplier_details'].items():
            print(f"  {supplier}: total stock = {details['total_number']}, drugs = {details['list_of_drugs']}")
        print("\nCounts (form, category, supplier):")
        print(supp_sum['counts'])

        print("\n--- total_inventory_value() ---")
        total = drug_mgr.total_inventory_value()
        print(f"Total inventory value: ${total:,.2f}")
        by_supp = drug_mgr.total_inventory_value(by_supplier=True)
        print("By supplier:")
        for supp, val in by_supp.items():
            print(f"  {supp}: ${val:,.2f}")

        print("\n--- description() ---")
        print(drug_mgr.description())
    except Exception as e:
        print(f"Error in Drug class tests: {e}")

# ==================== Test Inventory class ====================
def test_inventory_class(filepath):
    print_header("Testing Inventory Class")
    try:
        # Load data using load_stock (or directly read)
        df = load_stock(filepath)  # ensures same data treatment
        inventory = Inventory(df)

        print("Inventory instance created. Total items:", inventory.total_items)

        print("\n--- get_low_stock() ---")
        low = inventory.get_low_stock()
        print("Low stock drugs (within 15 below reorder):")
        if low.empty:
            print("  None")
        else:
            for _, row in low.iterrows():
                print(f"  {row['drug_name']}: {row['quantity_in_stock']} (reorder {row['reorder_level']})")

        print("\n--- get_re_stock() ---")
        re = inventory.get_re_stock()
        print("Drugs needing restock soon (≤ reorder+20):")
        if re.empty:
            print("  None")
        else:
            for _, row in re.iterrows():
                print(f"  {row['drug_name']}: {row['quantity_in_stock']} (reorder {row['reorder_level']})")

        print("\n--- stock_summary() ---")
        summary = inventory.stock_summary()
        print("Most stock:", summary['most_stock']['drug_name'], "with", summary['most_stock']['quantity_in_stock'])
        print("Least stock:", summary['least_stock']['drug_name'], "with", summary['least_stock']['quantity_in_stock'])
        print("Latest expiry:", summary['latest_expiry']['drug_name'], "on", summary['latest_expiry']['expiry_date'].date())
        print("Oldest expiry:", summary['oldest_expiry']['drug_name'], "on", summary['oldest_expiry']['expiry_date'].date())

        print("\n--- check_expiry() ---")
        expired = inventory.check_expiry()
        print("Expired drugs:", len(expired))
        if not expired.empty:
            for _, row in expired.iterrows():
                print(f"  {row['drug_name']} expired on {row['expiry_date'].date()}")

        print("\n--- near_expiry() (30 days) ---")
        near = inventory.near_expiry(30)
        print("Drugs expiring within 30 days:", len(near))
        if not near.empty:
            for _, row in near.iterrows():
                print(f"  {row['drug_name']} expires {row['expiry_date'].date()}")

        print("\n--- expiry_loss_forecast() (30, 60, 90 days) ---")
        for days in [30, 60, 90]:
            loss = inventory.expiry_loss_forecast(days)
            print(f"  Loss forecast for {days} days: ${loss:,.2f}")

        print("\n--- restock_report() (first 10 lines) ---")
        report = inventory.restock_report()  # returns string
        lines = report.split("\n")
        for line in lines[:10]:
            print(line)
        print("...")

        print("\n--- expiry_report() (60 days, first 10 lines) ---")
        exp_report = inventory.expiry_report(days=60)
        lines = exp_report.split("\n")
        for line in lines[:10]:
            print(line)
        print("...")
    except Exception as e:
        print(f"Error in Inventory class tests: {e}")

# ==================== Test DrugItem class ====================
def test_drugitem_class(filepath):
    print_header("Testing DrugItem Class")
    try:
        df = load_stock(filepath)
        # Pick up to 5 rows to test (first few)
        for i in range(min(5, len(df))):
            row = df.iloc[i]
            drug = DrugItem(row)
            print(f"\n--- DrugItem {i+1}: {drug.drug_name} ---")
            print("  is_expired():", drug.is_expired())
            print("  days_to_expiry():", drug.days_to_expiry())
            print("  needs_reorder():", drug.needs_reorder())
            print("  value(): ${:.2f}".format(drug.value()))
            print("  expiry_status():", drug.expiry_status())
            print("  reorder_status():", drug.reorder_status())
            print("  value_description():", drug.value_description())
            print("  stock_status():", drug.stock_status())
            print("  full_report():\n" + "\n".join("    " + line for line in drug.full_report().split("\n")))
            # Uncomment to see full dict keys
            # print("  info_dict() keys:", list(drug.info_dict().keys()))
    except Exception as e:
        print(f"Error in DrugItem tests: {e}")

# ==================== Test generate functions ====================
def test_generate_functions(filepath):
    print_header("Testing generate_restock_report and generate_expiry_report")
    try:
        df = load_stock(filepath)
        inventory = Inventory(df)

        restock_str = generate_restock_report(inventory)
        expiry_str = generate_expiry_report(inventory)

        print("generate_restock_report returned a string of length", len(restock_str))
        print("\n--- FULL RESTOCK REPORT (from generate_restock_report) ---")
        print(restock_str)

        print("\n" + "-"*60)
        print("generate_expiry_report returned a string of length", len(expiry_str))
        print("\n--- FULL EXPIRY REPORT (from generate_expiry_report) ---")
        print(expiry_str)

    except Exception as e:
        print(f"Error in generate functions tests: {e}")

# ==================== Run all tests ====================
if __name__ == "__main__":
    # Uncomment the next line if your code is in a separate module
    # from your_module import load_stock, Drug, Inventory, DrugItem, generate_restock_report, generate_expiry_report

    # Check if file exists
    import os
    if not os.path.exists(filepath):
        print(f"ERROR: File '{filepath}' not found. Please update the filepath variable.")
    else:
        test_load_stock(filepath)
        test_drug_class(filepath)
        test_inventory_class(filepath)
        test_drugitem_class(filepath)
        test_generate_functions(filepath)

        print_header("All tests completed")
        print("Review the output above for any errors or unexpected results.")


Testing load_stock
File loaded successfully. Shape: (102, 9)
Columns: ['drug_id', 'drug_name', 'category', 'quantity_in_stock', 'reorder_level', 'unit_price_usd', 'expiry_date', 'form', 'supplier']
Missing values after load_stock: 0
Data sorted by supplier? (check first few):
      supplier           drug_name
79  Supplier A  Magnesium Sulphate
32  Supplier A             Codeine
78  Supplier A   Calcium Gluconate
40  Supplier A        Theophylline
28  Supplier A        Penicillin V

Testing Drug Class
Drug instance created. Total drugs: 102

--- price_summary() ---
Most expensive: Oseltamivir ($49.91)
Cheapest: Naproxen ($0.69)

--- supplier_summary() ---
Supplier details:
  Supplier A: total stock = 6528, drugs = ['Metformin', 'Diazepam', 'Folic Acid', 'Metronidazole', 'Ceftriaxone', 'Penicillin V', 'Codeine', 'Aspirin', 'Prednisolone', 'Theophylline', 'Glibenclamide', 'Metoprolol', 'Captopril', 'Hydrochlorothiazide', 'Chlorpromazine', 'Fluoxetine', 'Vitamin B Complex', 'Calcium Gluc